# Optional Advanced L13a: Markov Models and Hidden Markov Models
In this lecture, we will explore the concepts of Markov Models and Hidden Markov Models (HMMs), which are widely used in various fields such as natural language processing, speech recognition, and bioinformatics. We are going to apply these concepts to practical problems in quantitative finance.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
> * __Construct discrete Markov models with state spaces and transition matrices:__ Define state spaces for systems, build transition probability matrices satisfying row normalization constraints, and apply the Markov property to model memoryless stochastic processes.
> * __Compute stationary distributions using power iteration methods:__ Apply iterative algorithms to estimate stationary distributions from transition matrices, implement convergence criteria based on norm differences, and interpret stationary distributions as long-run probabilities.
> * __Implement Hidden Markov Models with emission matrices:__ Define hidden state spaces and observable output spaces, construct emission probability matrices linking hidden states to observations, and generate synthetic sequences by sampling from transition and emission distributions.


Let's get started!
___

## Examples
Today, we will use the following examples to illustrate key concepts:

> [▶ Analyze a three-state hidden Markov model for mood](CHEME-5660-L13a-Example-HiddenMarkovModels-Fall-2026.ipynb). In this example, we will analyze a simple hidden Markov model (HMM) that represents a person's mood (happy, sad, neutral) based on observable behaviors (smiling, frowning, neutral expression). We will set up the HMM, compute the most likely sequence of moods given a series of observed behaviors, and discuss how this model can be applied in real-world scenarios.

> [▶ Analyze an in-sample hidden Markov model (HMM) of the daily SPY growth rate](CHEME-5660-L13a-Example-HMM-SPY-IS-Fall-2026.ipynb). In this example, we will analyze an in-sample hidden Markov model (HMM) applied to the growth rate of the SPY ETF. We will set up the HMM, estimate its parameters using historical SPY growth rate data, and evaluate its performance in capturing the stylized facts governing financial time series.
___

## Markov Chains
A discrete Markov model is a stochastic (random) process that describes the transitions between states based on defined probabilities. Formally, a Markov model is determined by the tuple $\mathcal{M} = (\mathcal{S},\mathbf{P})$, where $\mathcal{S}$ represents the set of states in the model and $\mathbf{P}$ is the transition matrix.

### The state space $\mathcal{S}$
The state space $\mathcal{S}$ is the set of all possible values a system can assume. For example, if a Markov chain can be in the states $\left\{1,2,3\right\}$, then $\mathcal{S} \equiv \left\{1,2,3\right\}$. What do these states represent? The answer depends on the problem. Here are two concrete examples:
* __Letters in words__ $\mathcal{S} \equiv \left\{a,b,c,\dotsc,z\right\}$: If the state space $\mathcal{S}$ is defined as the alphabet, we could develop a Markov model to generate $n$ words, each of length $l$ characters, that start with a specific letter such as $t$.
* __Investor moods__ $\mathcal{S} \equiv \left\{\text{bullish},\text{neutral},\text{bearish}\right\}$: In this case, the state space is defined as three possible moods an investor can experience. Using this type of $\mathcal{S}$, we could build a Markov model to simulate how an investor's mood changes as they watch the market, the news, and other factors.

Next, let's look at the Markov property and the transition matrix $\mathbf{P}$.

### The Markov Property
The Markov property describes a random process whose next-state probabilities depend only on its present state. Choosing a state that contains the information needed to predict the next step lets us describe the process using one-step transition probabilities.

> __Definition:__ Let $X_n$ denote the state of a stochastic process at time step $n$. The process satisfies the **Markov property** if the following equality holds for every $n\geq 0$ and every state history with positive probability:
>
> $$
> P(X_{n+1}=s_{n+1}\mid X_0=s_0,\ldots,X_n=s_n)
> =P(X_{n+1}=s_{n+1}\mid X_n=s_n).
> $$
>
> Given the present state, the next state is conditionally independent of the past states. This lets us update the state distribution without storing the full history of the process.

#### One-Step Transition Probabilities
For a finite state space $\mathcal{S}=\{s_1,\ldots,s_N\}$, the probability of moving from state $s_i$ to state $s_j$ in one step is given by:

$$
p_{ij}=P(X_{n+1}=s_j\mid X_n=s_i).
$$

We assume the process is __time-homogeneous__, meaning these probabilities stay constant over time. Thus, $p_{ij}$ does not depend on $n$, and the same transition matrix applies at every step.

#### Why the Markov Property Matters
The Markov property simplifies our modeling work: instead of tracking the entire history of a process, we only need to know the present state. Without this property, the number of possible historical sequences would grow exponentially. With it, the problem becomes tractable and easier to work with.

### The Transition Matrix
We collect the one-step probabilities in the transition matrix $\mathbf{P}\in\mathbb{R}^{N\times N}$. Row $i$ describes the next-state distribution when the current state is $s_i$:

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<div>
    <center>
        <img class="course-diagram" src="figs/Fig-TransitionMatrix-Schematic.svg" width="680" style="max-width:100%; height:auto;" alt="Schematic of a transition matrix drawn as a grid of dots inside square brackets. A shaded row is labeled rows equal current states, with the note sum equals 1. A shaded column is labeled columns equal future states. A red dot in row one, column two is labeled p12, and a yellow dot on the diagonal is labeled pii, a self-loop."/>
    </center>
</div>

The transition matrix has the following properties:
* Every entry is non-negative: $p_{ij}\geq 0$.
* Rows represent the current states, and columns represent the next states (our convention). Entry $p_{ij}$ is the probability of moving from $s_i$ to $s_j$.
* Every row sums to one: $\sum_{j=1}^{N}p_{ij}=1$. Starting from any state, the chain must move to one of the states in $\mathcal{S}$, including the possibility of staying where it is.

> __Example:__ Consider a 2-state Markov model of the weather with states $\mathcal{S} = \{\text{Sunny}, \text{Rainy}\}$ and the transition matrix:
>
> $$\mathbf{P} = \begin{bmatrix} 0.8 & 0.2 \\ 0.4 & 0.6 \end{bmatrix}$$
>
> Row 1 is Sunny and row 2 is Rainy. If today is Sunny, there is an 80% chance tomorrow is Sunny and a 20% chance it is Rainy. If today is Rainy, there is a 40% chance tomorrow is Sunny and a 60% chance it is Rainy.
>
> Starting from the initial distribution $\pi_0 = [1, 0]$, which says today is Sunny, the distribution for tomorrow is:
>
> $$\pi_1 = \pi_0 \mathbf{P} = [1, 0]\begin{bmatrix} 0.8 & 0.2 \\ 0.4 & 0.6 \end{bmatrix} = [0.8, 0.2]$$

#### Estimating the transition matrix from data
In finance, we estimate $\mathbf{P}$ from a sequence of observed states $X_{1},\dotsc,X_{T}$. Let $n_{ij}$ be the number of days in state $s_{i}$ followed by a day in state $s_{j}$. The estimate of $p_{ij}$ is the share of the days in state $s_{i}$ that moved to state $s_{j}$:
$$
\hat{p}_{ij} = \frac{n_{ij}}{\sum_{k=1}^{N} n_{ik}}
$$
Each row of $\hat{\mathbf{P}}$ sums to one by construction. This counting estimate is the maximum likelihood estimate of $\mathbf{P}$, and we will use it to estimate the transition matrix of the market generator.

### The Stationary Distribution
The stationary distribution $\bar{\pi}$ of a Markov chain with transition matrix $\mathbf{P}$ is a probability distribution that remains unchanged as the system evolves: $\bar{\pi} = \bar{\pi}\mathbf{P}$. It is the long-run share of time the chain spends in each state, and it is where a generator starts its paths.

Software often computes $\bar{\pi}$ by solving the linear equations $\bar{\pi} = \bar{\pi}\mathbf{P}$ and $\sum_{i}\bar{\pi}_{i} = 1$ directly. The power iteration method instead applies the transition matrix to an initial distribution until the distribution stops changing:

$$
\begin{align*}
\pi_1 &= \pi_0 \mathbf{P} \\
\pi_2 &= \pi_1 \mathbf{P} = \pi_0 \mathbf{P}^2 \\
\pi_3 &= \pi_2 \mathbf{P} = \pi_0 \mathbf{P}^3 \\
&\vdots \\
\pi_k &= \pi_{k-1} \mathbf{P} = \pi_0 \mathbf{P}^k
\end{align*}
$$

__Initialize__: Given an ergodic transition matrix $\mathbf{P}$ with $N$ states, a tolerance $\text{tol}>0$, and a maximum number of iterations $T$, initialize the iteration counter $t\gets 1$, the uniform initial distribution $\pi \gets \left[1/N,\ldots,1/N\right]$, and $\texttt{converged}\gets\texttt{false}$.

While not $\texttt{converged}$ __do__:
1. __Compute:__ Set $\pi^{\prime} \gets \pi\mathbf{P}$.
2. __Check whether to stop:__
    - If $\lVert \pi^{\prime} - \pi \rVert_{1} \leq \text{tol}$, __then__ set $\texttt{converged}\gets\texttt{true}$ and return $\bar{\pi}\gets\pi^{\prime}$.
    - Otherwise, if $t\geq{T}$, __then__ return $\pi^{\prime}$ with $\texttt{converged}=\texttt{false}$ and a warning that the maximum number of iterations was reached without convergence.
3. __Update the distribution:__ Set $\pi\gets\pi^{\prime}$.
4. __Increment the iteration counter:__ Set $t\gets{t+1}$.

__Output__: The stationary distribution $\bar{\pi}$.

The convergence rate depends on the spectral gap (difference between the largest and second-largest eigenvalue magnitudes of $\mathbf{P}$), with larger gaps enabling faster convergence.

__Notes:__
1. The algorithm assumes $\mathbf{P}$ is ergodic (irreducible and aperiodic). Such a chain has exactly one stationary distribution, and power iteration reaches it from any starting distribution.
2. The $\pi^{\prime} \gets \pi\mathbf{P}$ step is a left vector-matrix multiplication: $\pi$ is a row vector, and the new probability of state $s_{j}$ is $\pi^{\prime}_{j} = \sum_{i}\pi_{i}p_{ij}$.
___

## Hidden Markov Models (HMMs)
Next, let's explore hidden Markov models (HMMs).

> __Definition:__
>
> In a discrete hidden Markov model (HMM), the states are not directly observable. Instead, we observe outputs (emissions) whose probabilities depend on the hidden state. An HMM is characterized by the tuple $\mathcal{M} = (\mathcal{S},\mathcal{O},\mathbf{P},\mathbf{E})$:
> * __State Space ($\mathcal{S}$)__: The set of the $|\mathcal{S}|$ hidden states that the system can be in.
> * __Observation Space ($\mathcal{O}$)__: The set of the $|\mathcal{O}|$ observable states (emissions).
> * __Transition Matrix ($\mathbf{P}$)__: The matrix $\mathbf{P}\in\mathbb{R}^{|\mathcal{S}|\times|\mathcal{S}|}$ of the probabilities of moving between hidden states.
> * __Emission Matrix ($\mathbf{E}$)__: The matrix $\mathbf{E}\in\mathbb{R}^{|\mathcal{S}|\times|\mathcal{O}|}$ of the probabilities of observing each observable state given a hidden state.

For every hidden state $s_{i}$, the transition probabilities and the emission probabilities each sum to one:
$$
\begin{align*}
p_{ij} &= P(X_{n+1} = s_j \mid X_{n} = s_i), & \sum_{j=1}^{|\mathcal{S}|} p_{ij} &= 1\\
e_{ij} &= P(O_{n} = o_j \mid X_{n} = s_i), & \sum_{j=1}^{|\mathcal{O}|} e_{ij} &= 1
\end{align*}
$$

### Sampling Algorithm
Let's outline a simple sampling algorithm for generating random sequences from an HMM.

__Initialize__: Given an HMM defined by $\mathcal{M} = (\mathcal{S},\mathcal{O},\mathbf{P},\mathbf{E})$, the number of time steps $T$, and the stationary distribution $\bar{\pi}$. Draw an initial hidden state $X_0$ from the initial distribution: $X_0\sim \bar{\pi}$.

For each time step $n = 1,2,\dots,T$ __do__:
1. __Hidden state:__ Sample the next hidden state $X_n$ from the $\texttt{categorical}$ distribution $\mathbb{P}_{H}(...)$ defined by the previous state $X_{n-1}$ and the transition matrix $\mathbf{P}$: $X_n \sim \mathbb{P}_{H}(X_n \mid X_{n-1})$.
2. __Emission:__ Sample the observable state $O_n$ from the $\texttt{categorical}$ distribution $\mathbb{P}_{E}(...)$ defined by the current hidden state $X_n$ and the emission matrix $\mathbf{E}$: $O_n \sim \mathbb{P}_{E}(O_n \mid X_n)$.
3. __Store:__ Store $X_n$ and $O_n$.

__Output__: The sequences of hidden states $\{X_1, X_2, \ldots, X_T\}$ and observable states $\{O_1, O_2, \ldots, O_T\}$. The initial state $X_0$ is used to generate the sequence but is not included in the output.

> __What is a categorical distribution?__ A categorical distribution describes a random variable that takes one of $k$ categories, each with its own probability. It generalizes the Bernoulli distribution, which has two categories. Each row of $\mathbf{P}$ and of $\mathbf{E}$ is a categorical distribution, and the algorithm samples from these rows at each time step.

#### Continuous emissions
For a market generator, the observation is a growth rate, a real number rather than one of a few categories. In place of a row of $\mathbf{E}$, each hidden state $s_{k}$ has an emission __distribution__, a probability density of the growth rate given the state. Step 2 of the sampling algorithm then draws the growth rate from the emission distribution of the current state:
$$
g_{n} \mid X_{n} = s_{k} \;\sim\; \text{emission distribution of } s_{k}
$$
The hidden state sets how volatile the market is today, and the emission turns the state into a growth rate.

Let's look at an example of an HMM in action for modeling the growth rate distribution of the SPY ETF.

> __Example:__
>
> [▶ Analyze an in-sample hidden Markov model (HMM) of the daily SPY growth rate](CHEME-5660-L13a-Example-HMM-SPY-IS-Fall-2026.ipynb). In this example, we will analyze an in-sample hidden Markov model (HMM) of the growth rate of the SPY ETF. We will set up the HMM, estimate its parameters using historical SPY growth rate data, and evaluate its performance in capturing the stylized facts governing financial time series.
___

## Summary
This lecture introduces Markov models and Hidden Markov Models as frameworks for modeling stochastic processes with memoryless state transitions.

> __Key Takeaways:__
>
> * __Markov models encode memoryless state transitions through transition matrices:__ The Markov property states that future states depend only on the present state. Transition matrices encode one-step transition probabilities between states, with rows summing to one. These models simplify analysis by eliminating dependence on historical sequences.
> * __Stationary distributions represent long-run state probabilities:__ Power iteration methods compute stationary distributions by repeatedly applying transition matrices to initial distributions until convergence. The stationary distribution satisfies the equation $\bar{\pi} = \bar{\pi}\mathbf{P}$ and remains unchanged under further transitions. Convergence speed depends on the spectral gap of the transition matrix.
> * __Hidden Markov Models separate hidden states from observable outputs:__ HMMs extend Markov models by introducing emission matrices that link hidden states to observable outputs. The state space contains hidden states, while the observation space contains observable emissions. Sampling algorithms generate sequences by drawing hidden states from transition distributions and outputs from emission distributions.


Markov models and Hidden Markov Models provide tools for modeling sequential data and stochastic processes across finance, language processing, and other domains.
___